# Goodreads Data Collection

## Objective

The objective of this notebook is to obtain Goodreads ratings for the identified book adaptations and integrate them with the adaptations dataset.

## Methodology

1. A Goodreads dataset obtained from Kaggle was used as the primary source.
2. Extensive preprocessing was performed to standardize titles and author names.
3. Exact and fuzzy matching techniques were applied to match books across datasets.
4. Unmatched books were manually verified and collected from Goodreads to ensure complete coverage.


# GoodReads Kaggle Dataset

In [1]:
import pandas as pd

In [2]:
amazon_movies = pd.read_csv("../data/movies/final/amazon_movies.csv")
amazon_series = pd.read_csv("../data/series/final/amazon_series.csv")

netflix_movies = pd.read_csv("../data/movies/final/netflix_movies.csv")
netflix_series = pd.read_csv("../data/series/final/netflix_series.csv")

In [3]:
amazon_movies['platform'] = 'Amazon'
amazon_series['platform'] = 'Amazon'

netflix_movies['platform'] = 'Netflix'
netflix_series['platform'] = 'Netflix'

In [4]:
adaptations_df = pd.concat([amazon_series, amazon_movies, netflix_series, netflix_movies],ignore_index=True)

In [5]:
print(adaptations_df[['Title', 'platform']].head())

                                       Title platform
0                                    Reacher   Amazon
1                          The Terminal List   Amazon
2  The Lord of the Rings: The Rings of Power   Amazon
3                                      Cross   Amazon
4                              We Were Liars   Amazon


In [6]:
adaptations_df.to_csv('../data/books/processed/adaptations.csv', index=False)

In [7]:
df = pd.read_csv("../data/books/raw/books.csv", engine='python', on_bad_lines='skip')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11119 entries, 0 to 11118
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11119 non-null  int64  
 1   title               11119 non-null  str    
 2   authors             11119 non-null  str    
 3   average_rating      11119 non-null  float64
 4   isbn                11119 non-null  str    
 5   isbn13              11119 non-null  int64  
 6   language_code       11119 non-null  str    
 7     num_pages         11119 non-null  int64  
 8   ratings_count       11119 non-null  int64  
 9   text_reviews_count  11119 non-null  int64  
 10  publication_date    11119 non-null  str    
 11  publisher           11119 non-null  str    
dtypes: float64(1), int64(5), str(6)
memory usage: 1.0 MB


In [8]:
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


In [9]:
df.columns = df.columns.str.strip()

In [10]:
import re
import pandas as pd

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)     # remove extra spaces

    return text

df['search_key'] = df['title'].apply(clean_text)

In [11]:
df['primary_author'] = (df['authors'].str.split('/').str[0])

In [12]:
df['book_author'] = (df['primary_author'].apply(clean_text))

In [13]:
df[['authors', 'primary_author', 'book_author']].head()

,authors,primary_author,book_author
0,J.K. Rowling/Mary GrandPré,J.K. Rowling,jk rowling
1,J.K. Rowling/Mary GrandPré,J.K. Rowling,jk rowling
2,J.K. Rowling,J.K. Rowling,jk rowling
3,J.K. Rowling/Mary GrandPré,J.K. Rowling,jk rowling
4,J.K. Rowling/Mary GrandPré,J.K. Rowling,jk rowling


In [14]:
df.duplicated(subset=['search_key', 'book_author']).sum()

np.int64(784)

In [15]:
duplicates = df[df.duplicated(
    subset=['search_key', 'book_author'],
    keep=False
)]

duplicates.sort_values(
    ['search_key', 'book_author']
).head(20)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,search_key,primary_author,book_author
1585,5477,1984,George Orwell/Erich Fromm,4.18,0451516753,9780451516756,eng,268,1322,121,7/1/1981,Signet Classics,1984,George Orwell,george orwell
5284,19100,1984,George Orwell/Mauricio Molina/Miguel Martínez ...,4.18,9685270880,9789685270885,spa,301,451,42,8/1/2002,Lectorum,1984,George Orwell,george orwell
3628,13137,1st to Die (Women's Murder Club #1),James Patterson,4.08,0446696617,9780446696616,eng,424,269826,5220,5/20/2005,Grand Central Publishing,1st to die,James Patterson,james patterson
8733,33681,1st To Die (The Women's Murder Club #1),James Patterson,4.08,0613608704,9780613608701,eng,471,24,4,2/1/2002,Turtleback Books,1st to die,James Patterson,james patterson
3627,13136,2nd Chance (Women's Murder Club #2),James Patterson/Andrew Gross,4.04,0446696633,9780446696630,eng,400,78780,2118,5/20/2005,Grand Central Publishing,2nd chance,James Patterson,james patterson
5732,21426,2nd Chance (Women's Murder Club #2),James Patterson/Melissa Leo/Jeremy Piven/Andre...,4.04,1594831165,9781594831164,eng,5,28,4,2/27/2006,Little Brown & Company,2nd chance,James Patterson,james patterson
1537,5327,A Christmas Carol,Charles Dickens/P.J. Lynch,4.05,0763631205,9780763631208,en-CA,160,4359,353,9/12/2006,Candlewick Press,a christmas carol,Charles Dickens,charles dickens
1538,5328,A Christmas Carol,Charles Dickens,4.05,1580495796,9781580495790,eng,112,6671,490,1/1/2012,Penguin Books,a christmas carol,Charles Dickens,charles dickens
2845,10572,A Clash of Kings (A Song of Ice and Fire #2),George R.R. Martin,4.41,0553381695,9780553381696,eng,969,638766,16535,5/28/2002,Bantam,a clash of kings,George R.R. Martin,george rr martin
3729,13504,A Clash of Kings (A Song of Ice and Fire #2),George R.R. Martin/Roy Dotrice,4.41,073930870X,9780739308707,eng,0,113,12,2/17/2004,Random House Audio,a clash of kings,George R.R. Martin,george rr martin


In [16]:
df = (df.sort_values('ratings_count', ascending=False).drop_duplicates(subset=['search_key', 'book_author'],keep='first'))

In [17]:
df.duplicated(subset=['search_key', 'book_author']).sum()

np.int64(0)

In [18]:
df.head(10)[['title', 'authors', 'search_key', 'book_author']]

,title,authors,search_key,book_author
10333,Twilight (Twilight #1),Stephenie Meyer,twilight,stephenie meyer
1696,The Hobbit or There and Back Again,J.R.R. Tolkien,the hobbit or there and back again,jrr tolkien
1462,The Catcher in the Rye,J.D. Salinger,the catcher in the rye,jd salinger
307,Angels & Demons (Robert Langdon #1),Dan Brown,angels demons,dan brown
3,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,harry potter and the prisoner of azkaban,jk rowling
4414,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling/Mary GrandPré,harry potter and the chamber of secrets,jk rowling
1,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,harry potter and the order of the phoenix,jk rowling
23,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. Tolkien,the fellowship of the ring,jrr tolkien
2113,Animal Farm,George Orwell/Boris Grabnar/Peter Škerl,animal farm,george orwell
0,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,harry potter and the halfblood prince,jk rowling


In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 10335 entries, 10333 to 5722
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              10335 non-null  int64  
 1   title               10335 non-null  str    
 2   authors             10335 non-null  str    
 3   average_rating      10335 non-null  float64
 4   isbn                10335 non-null  str    
 5   isbn13              10335 non-null  int64  
 6   language_code       10335 non-null  str    
 7   num_pages           10335 non-null  int64  
 8   ratings_count       10335 non-null  int64  
 9   text_reviews_count  10335 non-null  int64  
 10  publication_date    10335 non-null  str    
 11  publisher           10335 non-null  str    
 12  search_key          10335 non-null  str    
 13  primary_author      10335 non-null  object 
 14  book_author         10335 non-null  str    
dtypes: float64(1), int64(5), object(1), str(8)
memory usage: 1.3+ MB


In [20]:
goodreads_df = df[['title', 'authors', 'search_key', 'book_author', 'average_rating']]

In [21]:
goodreads_df.info()

<class 'pandas.DataFrame'>
Index: 10335 entries, 10333 to 5722
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           10335 non-null  str    
 1   authors         10335 non-null  str    
 2   search_key      10335 non-null  str    
 3   book_author     10335 non-null  str    
 4   average_rating  10335 non-null  float64
dtypes: float64(1), str(4)
memory usage: 484.5 KB


In [22]:
adaptations_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               267 non-null    str    
 1   Premiere            267 non-null    str    
 2   Rated               251 non-null    str    
 3   imdb_rating         267 non-null    float64
 4   imdb_votes          267 non-null    float64
 5   imdb_genre          267 non-null    str    
 6   imdb_year           267 non-null    int64  
 7   imdb_type           267 non-null    str    
 8   is_book_adaptation  267 non-null    str    
 9   book_title          267 non-null    str    
 10  book_author         267 non-null    str    
 11  search_key          267 non-null    str    
 12  platform            267 non-null    str    
dtypes: float64(2), int64(1), str(10)
memory usage: 27.2 KB


In [23]:
adaptations_df = adaptations_df.merge(
    goodreads_df[['search_key', 'book_author', 'average_rating']],
    on=['search_key', 'book_author'],
    how='left'
)

adaptations_df.rename(columns={'average_rating': 'goodreads_rating'},inplace=True)

In [24]:
print("Matched:", adaptations_df['goodreads_rating'].notna().sum())
print("Unmatched:", adaptations_df['goodreads_rating'].isna().sum())

Matched: 14
Unmatched: 253


In [25]:
title_matches = adaptations_df.merge(
    goodreads_df[['search_key', 'average_rating']],
    on='search_key',
    how='left'
)

print("Title-only matches:",
      title_matches['average_rating'].notna().sum())

Title-only matches: 21


In [26]:
common_titles = set(adaptations_df['search_key']) & set(goodreads_df['search_key'])

print("Common titles:", len(common_titles))

Common titles: 18


In [27]:
from rapidfuzz import process, fuzz

goodreads_titles = goodreads_df['search_key'].tolist()

def find_match(title, threshold=90):
    match = process.extractOne(
        title,
        goodreads_titles,
        scorer=fuzz.ratio
    )

    if match and match[1] >= threshold:
        return match[0]

    return None

adaptations_df['matched_title'] = (
    adaptations_df['search_key']
    .apply(find_match)
)

In [28]:
print("Matched:", adaptations_df['matched_title'].notna().sum())
print("Unmatched:", adaptations_df['matched_title'].isna().sum())

Matched: 29
Unmatched: 238


## Challenges and Limitations

During the process of integrating book ratings, an attempt was made to automatically match the adaptation dataset with a Goodreads dataset obtained from Kaggle. Both datasets underwent extensive preprocessing, including text normalization, removal of punctuation, standardization of author names, and duplicate handling. Exact matching and fuzzy matching techniques were also explored to maximize the number of successful matches.

Despite these efforts, the Goodreads dataset provided limited coverage, resulting in a low match rate. This was primarily due to differences in book editions, title formats, and the limited size of the available Goodreads dataset.

To ensure completeness and maintain data quality, Goodreads ratings for the final set of book adaptations were manually verified and collected from the Goodreads website. Although this approach was more time-consuming, it ensured that each adaptation was associated with the correct book rating, thereby improving the reliability of the analysis.

This challenge highlights a common issue in real-world data science projects: integrating data from multiple sources often requires a combination of automated techniques and manual validation to achieve accurate results.


In [ ]:
ratings_df = adaptations_df[['Title','search_key', 'book_author']].copy()
ratings_df['goodreads_rating'] = ''
ratings_df.to_csv('../data/books/processed/manual_goodreads_ratings.csv',index=False)

## Final Dataset Preparation

A separate file, `ratings_and_final_check.csv`, was created to manually collect and verify Goodreads ratings for all book adaptations. During this process, additional validation was performed to ensure the accuracy of the dataset.

While manually reviewing the books, it was observed that some `search_key` values were incorrect and were subsequently updated. In addition, a few movies and series that had initially been identified as book adaptations were found to be incorrectly classified. Furthermore, some books did not have Goodreads ratings available.

To account for these cases, a new column, `keep`, was introduced with values `Y` and `N`, indicating whether a record should be retained for the final analysis.

After incorporating the manually collected Goodreads ratings and performing the necessary corrections, the updated adaptations dataset was saved as:

`../data/books/final/updated_adaptations.csv`
